# So sánh denoise GPU trên OCT Harvard-GF (DnCNN + SwinIR vs cổ điển)

Chạy **trên Colab GPU (Runtime → Change runtime type → T4/A100)**. Reproduce & mở rộng
`scripts/compare_denoise_methods.py` với **2 phương pháp deep-learning dùng GPU**:

| Phương pháp | Loại | Weights |
|---|---|---|
| `dncnn` | CNN residual (blind) | KAIR `dncnn_gray_blind.pth` (~2.7 MB) |
| `swinir` | Swin-Transformer grayDN, σ=25 | chính thức `004_grayDN_DFWB_s128w8_SwinIR-M_noise25.pth` (~123 MB) |

Các pp cổ điển (gaussian/median/tv/nlm/bm3d) chạy CPU trong
`skimage`/`bm3d`. `bm3d` rất chậm trên CPU Colab (2 core) — để `RUN_BM3D=False`
trừ khi bạn sẵn sàng chờ ~30-60 phút/volume. Output ghi vào `figures/denoise/`
trong repo đã clone (cache ở `%TEMP%/gf_denoise_weights` + `figures/denoise/cache/`).


## 1. Clone repo + cài thư viện

In [ ]:
!git clone --depth 1 https://github.com/Tqhuyen/glaucoma-thesis.git /content/glaucoma-thesis 2>/dev/null || git -C /content/glaucoma-thesis pull --ff-only 2>/dev/null || true
%cd /content/glaucoma-thesis
!pip install -q bm3d scikit-image matplotlib pandas requests huggingface_hub
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2. Import + kiểm tra device

In [ ]:
import os, sys, json, time

sys.path.insert(0, "scripts")
import numpy as np
import torch
import compare_denoise_methods as cdm
import denoise_torch as dtorch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("gpu methods importable:", dtorch.TORCH_OK)


## 3. Cấu hình thí nghiệm

- `STORE_RES=200` : data thô **200³** — denoise đúng độ phân giải lưu trữ, không resize.
- `VOLUMES`: id Harvard-GF (mặc định `2404` = volume đã dùng trong báo cáo).
- `METHODS`: bỏ `bm3d` khỏi danh sách nếu không muốn chờ; `dncnn`+`swinir` tự lên GPU.
- `PLANES`: mặt cắt hiển thị (None = tái dùng meta cũ hoặc mặc định x/y/z).


In [ ]:
STORE_RES = 200
VOLUMES = ["2404"]
METHODS = ["gaussian", "median", "tv", "nlm", "dncnn", "swinir"]
RUN_BM3D = False
if RUN_BM3D:
    METHODS.append("bm3d")
WORKERS = 2
PLANES = None
if set(METHODS) & cdm.TORCH_METHODS and not torch.cuda.is_available():
    raise RuntimeError(
        'GPU methods (dncnn/swinir) need CUDA. Runtime -> Change runtime type -> T4/A100, then rerun.'
    )
if torch.cuda.is_available():
    print('GPU OK:', torch.cuda.get_device_name(0))


## 4. Wandb (bắt buộc) — đăng nhập key

In [ ]:
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception as e:
        print("no WANDB_API_KEY:", e)


def init_wandb(run_name, config=None):
    if not os.environ.get("WANDB_API_KEY"):
        return None
    import wandb
    try:
        return wandb.init(project="glaucoma-thesis", name=run_name,
                          config=config or {}, id=run_name, resume="allow")
    except Exception as e:
        print("[wandb] init failed, continuing without cloud logging:", e)
        return None


RUN_NAME = f"denoise-gpu-{'_'.join(VOLUMES)}"
run = init_wandb(RUN_NAME, config={"volumes": VOLUMES, "methods": METHODS, "store_res": STORE_RES})
print("wandb run:", RUN_NAME if run else None)


## 5. Denoise + metric + ảnh so sánh

Lặp lại `compare_denoise_methods.denoise_volume` cho từng method (GPU methods batch
trên CUDA; mọi pp áp dụng 2D trên từng B-scan như pipeline cũ). Metric no-reference
(SNR/ENL/CNR/β) đo cùng vùng RNFL `[lo,hi)` & nền `[bl,lo)` như tài liệu
`figures/denoise/DENOISE_METHODS_COMPARISON.md`.


In [ ]:
def run_volume(stem, methods, planes=None):
    vol = cdm.load_volume(stem)
    dz = cdm.depth_axis(vol)
    volc = cdm.crop_lateral(vol, dz)
    lo, hi, bl, _ = cdm.band_indexes(volc, dz)
    band = cdm.stack_band(volc, dz, lo, hi).astype(np.float32).ravel()
    thresh = float(np.percentile(band, 60))
    planes = planes or (cdm.old_meta_planes(stem) or cdm.DEFAULT_PLANES)
    print(f"[vol {stem}] shape={vol.shape} dz={dz} rnfl=[{lo},{hi}) bg=[{bl},{lo}) "
          f"signal_thresh={thresh:.1f}", flush=True)

    volumes = {"original": vol}
    elapsed = {"original": 0.0}
    for name in methods:
        if name == "bm3d" and not cdm.HAVE_BM3D:
            print(f"[skip] {name} not importable (pip install bm3d)")
            continue
        if name in cdm.TORCH_METHODS and not dtorch.TORCH_OK:
            print(f"[skip] {name} needs torch")
            continue
        cache_path = os.path.join(cdm.DENOISE_CACHE, f"{stem}_{name}.npy")
        t0 = time.time()
        den, sec = cdm.denoise_volume(vol, name, workers=WORKERS, cache_path=cache_path)
        volumes[name] = den
        elapsed[name] = sec
        print(f"[denoise] {name} done in {max(sec, time.time()-t0):.1f}s", flush=True)

    met = {}
    for key, v in volumes.items():
        met[key] = cdm.volume_metrics(vol, v, dz, thresh)
        print(f"[metrics] {key:9s} SNR={met[key]['snr']:.3f} ENL={met[key]['enl']:.3f} "
              f"CNR={met[key]['cnr']:.3f} beta={met[key]['beta']:.3f}", flush=True)

    rows_img = [("Original (raw 200^3)", vol)] + [(cdm.row_label(n), volumes[n]) for n in volumes if n != "original"]
    fig_all = os.path.join(cdm.FIG_DIR, f"denoise_compare_{stem}_all.png")
    cdm.draw_grid(rows_img, planes, fig_all,
                  suptitle=(f"Harvard-GF volume data_{stem} (200^3) | speckle denoising | "
                            f"GPU: dncnn/swinir | identical planes & contrast"))
    for name in methods:
        if name in volumes:
            cdm.draw_grid([("Original (raw 200^3)", vol), (cdm.row_label(name), volumes[name])],
                          planes,
                          os.path.join(cdm.FIG_DIR, f"denoise_compare_{stem}_{name}.png"),
                          suptitle=f"data_{stem} | {cdm.row_label(name)}")
    table_rows = [("Original", met["original"]["snr"], met["original"]["enl"],
                   met["original"]["cnr"], met["original"]["beta"], 0.0)]
    for name in methods:
        if name in met:
            table_rows.append((name, met[name]["snr"], met[name]["enl"],
                               met[name]["cnr"], met[name]["beta"], elapsed[name]))
    csv_path = os.path.join(cdm.FIG_DIR, f"denoise_metrics_{stem}.csv")
    import csv as _csv
    with open(csv_path, "w", newline="") as fh:
        w = _csv.writer(fh)
        w.writerow(["method", "snr", "enl", "cnr", "beta", "time_s"])
        for r in table_rows:
            w.writerow(r)
    cdm.draw_metric_table(table_rows,
                          os.path.join(cdm.FIG_DIR, f"denoise_metrics_{stem}.png"),
                          title=f"Speckle denoising metrics - Harvard-GF volume data_{stem} (incl. GPU DL)")
    print("wrote", csv_path, "|", fig_all)
    return {"stem": stem, "dz": int(dz), "rnfl": [int(lo), int(hi)],
            "bg": [int(bl), int(lo)], "threshold": thresh,
            "metrics": met, "time_s": elapsed, "figures": {"all": fig_all}}


In [ ]:
results = [run_volume(s, list(METHODS), planes=PLANES) for s in VOLUMES]
print("\n".join(f"{s['stem']}: {len(s['metrics'])} methods compared" for s in results))


## 6. Drive sync (bắt buộc) - copy output lên Google Drive

Mọi figure/CSV sinh ra phải lưu vào Drive (xem AGENTS.md). Cell dưới mount Drive (bấm Authorize) rồi định nghĩa helper `sync_outputs()`; tự bỏ qua nếu không mount được.

In [ ]:
DRIVE_SYNC_DIR = os.environ.get("DRIVE_SYNC_DIR", "")


def mount_drive():
    global DRIVE_SYNC_DIR
    if DRIVE_SYNC_DIR:
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        DRIVE_SYNC_DIR = DRIVE_SYNC_DIR or "/content/drive/MyDrive/MasterBKDN/Thesis/denoise_gpu_figures"
        return True
    except Exception as e:
        print("[drive] mount skipped:", e)
        return False


def sync_outputs():
    if not DRIVE_SYNC_DIR:
        print("[drive] skipped - no DRIVE_SYNC_DIR (set env DRIVE_SYNC_DIR to force)")
        return 0
    import glob
    import shutil

    os.makedirs(DRIVE_SYNC_DIR, exist_ok=True)
    copied = 0
    for f in sorted(glob.glob(os.path.join(cdm.FIG_DIR, "denoise_*"))):
        shutil.copy2(f, os.path.join(DRIVE_SYNC_DIR, os.path.basename(f)))
        copied += 1
    print(f"[drive] synced {copied} files -> {DRIVE_SYNC_DIR}")
    return copied


mount_drive()

## 6. Log lên wandb + kết thúc run

In [ ]:
if run is not None:
    import wandb
    for res in results:
        stem = res["stem"]
        data = {}
        for k, v in res["metrics"].items():
            data[f"{stem}/{k}/snr"] = v["snr"]
            data[f"{stem}/{k}/enl"] = v["enl"]
            data[f"{stem}/{k}/cnr"] = v["cnr"]
            data[f"{stem}/{k}/beta"] = v["beta"]
            data[f"{stem}/{k}/time_s"] = res["time_s"].get(k, 0.0)
        run.log(data, step=1)
        for path in [res["figures"]["all"], os.path.join(cdm.FIG_DIR, f"denoise_metrics_{stem}.png")]:
            if os.path.exists(path):
                run.log({f"{stem}/img/{os.path.basename(path)}": wandb.Image(path)}, step=1)
        best_cnr = max(res["metrics"].items(), key=lambda kv: kv[1]["cnr"])[0]
        best_beta = max(res["metrics"].items(), key=lambda kv: kv[1]["beta"])[0]
        run.summary.update({f"{stem}/best_cnr": res["metrics"][best_cnr]["cnr"],
                            f"{stem}/best_beta": res["metrics"][best_beta]["beta"]})
    run.finish()
sync_outputs()
print("done - outputs under", cdm.FIG_DIR)


## 7. Kết quả & lưu ý

- **Runtime ước lượng trên T4**: DnCNN ~vài giây/volume; SwinIR ~1-4 phút/volume 200³;
  các pp CPU (tv/nlm...) vài giây; **BM3D CPU rất chậm** (~30-60 phút) — chỉ bật khi cần.
- Hai mô hình DL được train trên **nhiễu Gaussian** ([0,1], σ=25 / blind) nên số đo
  no-reference (SNR/ENL/CNR) chỉ tham khảo cho speckle — **hãy xem kèm ảnh**
  `denoise_compare_{volume}_all.png` và chỉ số `β` (giữ biên RNFL).
- Muốn chạy nhiều volume: sửa `VOLUMES = ["2404", "3294", ...]` ở cell 3 rồi Run all.
- File sinh ra trong `/content/glaucoma-thesis/figures/denoise/` (tải về / commit nếu cần).

Tài liệu tham khảo: Zhang et al. 2017 (DnCNN), Liang et al. 2021 (SwinIR),
Dabov et al. 2007 (BM3D); script nguồn `scripts/compare_denoise_methods.py` +
`scripts/denoise_torch.py`.


## 8. (Tuỳ chọn) Chọn mô hình tốt nhất -> build dataset denoise toàn bộ -> upload lên HF

Sau khi xem kết quả ở trên và **tự chọn được mô hình tốt hơn**, đổi `BEST_METHOD` ở cell dưới
rồi chạy cell này + cell cuối. Pipeline giống notebook `3d_glaucoma_bm3d_preprocess_upload.ipynb`:
stream từ zip Harvard-GF (tải 1 lần ~20 GB vào Colab) -> denoise từng volume bằng method đã chọn
(GPU: `dncnn`/`swinir`) -> ghi memmap `{Training,Validation,Test}_{volumes,labels}.npy` ->
**resume-safe** (đứt giữa chừng chạy lại là tiếp tục) -> upload folder lên HF dataset **private**.

Ước lượng thời gian full 3.300 volume (để khỏi sốc):
- `dncnn`  (GPU): ~0.5-1 ngày  |  `swinir` (GPU): ~3-5 ngày  |  `tv`/`nlm` (CPU): lâu
- `bm3d`   (CPU): **không khuyến nghị** (nhiều tuần) trừ khi bạn muốn chờ.
Giới hạn an toàn `ABORT_AFTER_HOURS` sẽ dừng sớm & lưu phần đã làm khi ước lượng vượt ngưỡng
(bạn có thể tăng rồi chạy lại để resume). `LIMIT_VOLUMES>0` để chạy thử N volume trước.

In [ ]:
# ---- [8a] cấu hình: đổi BEST_METHOD sau khi bạn đã chọn mô hình tốt nhất ----
BEST_METHOD = "dncnn"          # method sẽ dùng cho dataset: dncnn / swinir / tv / nlm / bm3d ...
OUT_REPO = ""                  # để trống = auto: <hf_user>/harvard-oct-glaucoma-200-<BEST_METHOD>
OUT_PRIVATE = True
STAGING = "/content/glaucoma_hf_200_" + BEST_METHOD
LIMIT_VOLUMES = 0              # 0 = toàn bộ; >0 = chỉ N volume đầu (thử nghiệm)
SAVE_EVERY = 25                # flush + ghi progress mỗi N volume
ABORT_AFTER_HOURS = 23.0       # dừng sớm khi ETA vượt ngưỡng (resume-safe)
GO = False                     # bật True ở cell cuối để chạy build + upload

HF_SOURCE = "harvardairobotics/Harvard-GF"
ZIP_FILE = "Dataset/dataset.zip"
CSV_FILE = "ReadMe/data_summary.csv"
SPLITS = ("Training", "Validation", "Test")
SPLIT_ALIAS = {"training": "Training", "validation": "Validation",
               "valid": "Validation", "test": "Test", "testing": "Test"}
RES = 200

assert BEST_METHOD in cdm.METHODS, f"BEST_METHOD phải thuộc {list(cdm.METHODS)}"
if BEST_METHOD in cdm.TORCH_METHODS:
    assert torch.cuda.is_available(), "BEST_METHOD là GPU method -> cần runtime GPU (T4/A100)"
if BEST_METHOD == "bm3d" and not cdm.HAVE_BM3D:
    raise RuntimeError("bm3d chưa cài được -> pip install bm3d")
print("BEST_METHOD =", BEST_METHOD, "| staging =", STAGING)

In [ ]:
# ---- [8b] build + upload (chạy khi GO = True) ----
if not GO:
    raise SystemExit("Set GO = True (cell 8a) rồi chạy lại cell này để build + upload.")

import io
import shutil
import csv as _csv
import zipfile as _zipfile
from huggingface_hub import hf_hub_download, HfApi


def _hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None


def _split_counts(meta, names):
    counts = {s: 0 for s in SPLITS}
    for n in names:
        counts[meta[os.path.splitext(os.path.basename(n))[0]][0]] += 1
    return counts


if not _hf_token():
    raise RuntimeError("HF_TOKEN missing (env hoặc Colab secret)")

csv_path = hf_hub_download(repo_id=HF_SOURCE, filename=CSV_FILE, repo_type="dataset")
zip_path = hf_hub_download(repo_id=HF_SOURCE, filename=ZIP_FILE, repo_type="dataset")
meta = {}
with open(csv_path, newline="") as fh:
    for r in _csv.DictReader(fh):
        split = SPLIT_ALIAS.get((r["use"] or "").strip().lower())
        if split is None:
            continue
        stem = os.path.splitext(os.path.basename(r["filename"]))[0]
        meta[stem] = (split, 1 if str(r["glaucoma"]).strip().lower() in ("yes", "1", "true") else 0)
with _zipfile.ZipFile(zip_path) as zf:
    names = sorted(n for n in zf.namelist() if n.endswith(".npz")
                   and os.path.splitext(os.path.basename(n))[0] in meta)
print("[8b] volumes:", len(names), "| splits:", _split_counts(meta, names), flush=True)

os.makedirs(STAGING, exist_ok=True)
counts = _split_counts(meta, names)
vols, labs = {}, {}
for s in SPLITS:
    if counts[s] == 0:
        continue
    mode = "r+" if os.path.exists(os.path.join(STAGING, f"{s}_volumes.npy")) else "w+"
    vols[s] = np.lib.format.open_memmap(os.path.join(STAGING, f"{s}_volumes.npy"), mode=mode,
                                        dtype=np.uint8, shape=(counts[s], 1, RES, RES, RES))
    labs[s] = np.lib.format.open_memmap(os.path.join(STAGING, f"{s}_labels.npy"), mode=mode,
                                        dtype=np.int64, shape=(counts[s],))
prog_path = os.path.join(STAGING, "progress.json")
done = set(json.load(open(prog_path)).get("stems", [])) if os.path.exists(prog_path) else set()
todo = [n for n in names if os.path.splitext(os.path.basename(n))[0] not in done]
if LIMIT_VOLUMES > 0:
    todo = todo[:LIMIT_VOLUMES]
filled = {}
for s in SPLITS:
    filled[s] = sum(1 for st in done if meta[st][0] == s)
print(f"[8b] resume: {len(done)}/{len(names)} done; {len(todo)} to do", flush=True)

t0, w0 = time.time(), len(done)
for i, entry in enumerate(todo):
    stem = os.path.splitext(os.path.basename(entry))[0]
    split, label = meta[stem]
    with _zipfile.ZipFile(zip_path) as zf:
        raw = np.load(io.BytesIO(zf.read(entry)))["oct_bscans"]
    den, sec = cdm.denoise_volume(raw, BEST_METHOD, workers=WORKERS, cache_path=None)
    vols[split][filled[split]] = den[None]
    labs[split][filled[split]] = label
    filled[split] += 1
    done.add(stem)
    if (len(done) - w0) % SAVE_EVERY == 0 or i == len(todo) - 1:
        for s in SPLITS:
            if s in vols:
                vols[s].flush(); labs[s].flush()
        with open(prog_path, "w") as fh:
            json.dump({"stems": sorted(done)}, fh)
        el = time.time() - t0
        rate = el / max(len(done) - w0, 1)
        eta_h = rate * max(len(todo) - i - 1, 0) / 3600
        print(f"[8b] {len(done)}/{len(names)} | {sec:.2f} s/vol | ETA {eta_h:.1f} h", flush=True)
        if eta_h > ABORT_AFTER_HOURS:
            print(f"[8b] ABORT: ETA {eta_h:.1f}h > {ABORT_AFTER_HOURS}h (resume-safe)", flush=True)
            break
else:
    for s in SPLITS:
        if s in vols:
            vols[s].flush(); labs[s].flush()
    with open(os.path.join(STAGING, "manifest.json"), "w") as fh:
        json.dump({"source": HF_SOURCE, "denoiser": BEST_METHOD,
                   "label": cdm.METHODS[BEST_METHOD]["label"],
                   "resolution": RES, "per_bscan_divide_255": True,
                   "layout": "glaucoma_all/{Training,Validation,Test}_{volumes,labels}.npy"}, fh, indent=2)
    print("[8b] build DONE", flush=True)

    owner = HfApi(token=_hf_token()).whoami()["name"]
    repo = OUT_REPO or f"{owner}/harvard-oct-glaucoma-200-{BEST_METHOD}"
    api = HfApi(token=_hf_token())
    api.create_repo(repo_id=repo, repo_type="dataset", private=OUT_PRIVATE, exist_ok=True)
    api.upload_folder(repo_id=repo, repo_type="dataset", folder_path=STAGING,
                      commit_message=f"Harvard-GF 200^3 denoised with {cdm.METHODS[BEST_METHOD]['label']}")
    print("uploaded:", "https://huggingface.co/datasets/" + repo, flush=True)